# Model cases

Runs the cases for the report on the base model (`model/microgrid.py`) plus add-ons (`model/extensions.py`).
Each case: build the base model → apply add-ons → solve → validate → add a row to the results table.
Equations: `docs/formulation.md`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT / "model"))
from params import Params, Y, V
from microgrid import build, validate, load_inputs
from extensions import add_outage, peak_day

In [2]:
def run(case, alpha=None, p=None):
    """Solve one case and return (summary row, unmet demand per representative year in kWh)."""
    p = p or Params()
    m, inv, ops, cost = build(p)
    m.Params.OutputFlag = 0
    if alpha is not None:
        add_outage(m, ops, cost, p, alpha, peak_day(load_inputs(p)))
    m.optimize()
    print(case); validate(p, ops)
    row = {"case": case, "Z ($M)": m.ObjVal / 1e6}
    row.update({f"{k} ($M)": float(e.getValue()) / 1e6 for k, e in cost.items()})
    for v in V:
        row[f"PV {v} (kW)"] = inv["xPV"][v].X
        row[f"Battery {v} (kWh)"] = inv["xB"][v].X
    unmet = {y: o["u"].X.sum() for y, o in ops.items()}
    row["Unmet, all rep. years (MWh)"] = sum(unmet.values()) / 1e3
    return row, unmet

## Resilience analysis
Outage on the peak-demand day (July 2) of every representative year. PV and battery are re-optimized in each run.
M = 10 CAD/kWh. Emissions per run will be added once carbon (Q10) is defined.

In [3]:
cases = [("Normal (α = 1)", None), ("Partial restriction (α = 0.5)", 0.5), ("Complete outage (α = 0)", 0.0)]
rows, unmet = [], {}
for name, alpha in cases:
    row, u = run(name, alpha)
    rows.append(row); unmet[name] = u
resilience = pd.DataFrame(rows).set_index("case").fillna(0)   # normal run has no C_unmet term
resilience.round(2)

Set parameter Username


Set parameter LicenseID to value 2889055


Academic license - for non-commercial use only - expires 2027-11-26


Normal (α = 1)
  2028: OK | balance error 3.6e-12 kWh | daily shift sum 3.9e-12 | SOC -0-0 of -0 kWh | unmet 0 kWh
  2031: OK | balance error 3.6e-12 kWh | daily shift sum 3.6e-12 | SOC 0-0 of 0 kWh | unmet 0 kWh
  2037: OK | balance error 7.3e-12 kWh | daily shift sum 5.0e-12 | SOC 0-0 of 0 kWh | unmet 0 kWh
  2043: OK | balance error 7.3e-12 kWh | daily shift sum 6.8e-12 | SOC 0-0 of 0 kWh | unmet 0 kWh
  2049: OK | balance error 7.3e-12 kWh | daily shift sum 8.6e-12 | SOC 0-0 of 0 kWh | unmet 0 kWh


Partial restriction (α = 0.5)
  2028: OK | balance error 3.6e-12 kWh | daily shift sum 3.9e-12 | SOC -0-0 of -0 kWh | unmet 18,981 kWh
  2031: OK | balance error 3.6e-12 kWh | daily shift sum 4.1e-12 | SOC 0-0 of 0 kWh | unmet 10,828 kWh
  2037: OK | balance error 7.3e-12 kWh | daily shift sum 4.1e-12 | SOC 0-0 of 0 kWh | unmet 66,951 kWh
  2043: OK | balance error 7.3e-12 kWh | daily shift sum 7.3e-12 | SOC 0-0 of 0 kWh | unmet 103,231 kWh
  2049: OK | balance error 7.3e-12 kWh | daily shift sum 8.6e-12 | SOC 0-0 of 0 kWh | unmet 169,091 kWh


Complete outage (α = 0)
  2028: OK | balance error 7.3e-12 kWh | daily shift sum 3.0e-11 | SOC -0-0 of -0 kWh | unmet 384,213 kWh
  2031: OK | balance error 3.6e-12 kWh | daily shift sum 4.1e-12 | SOC 0-0 of 0 kWh | unmet 354,376 kWh
  2037: OK | balance error 7.3e-12 kWh | daily shift sum 4.1e-12 | SOC 0-0 of 0 kWh | unmet 432,183 kWh
  2043: OK | balance error 7.3e-12 kWh | daily shift sum 7.3e-12 | SOC 0-0 of 0 kWh | unmet 468,463 kWh
  2049: OK | balance error 7.3e-12 kWh | daily shift sum 8.6e-12 | SOC 0-0 of 0 kWh | unmet 534,323 kWh


,Z ($M),C_inv ($M),C_OM ($M),C_rep ($M),C_RV ($M),C_grid ($M),PV 2028 (kW),Battery 2028 (kWh),PV 2040 (kW),Battery 2040 (kWh),"Unmet, all rep. years (MWh)",C_unmet ($M)
case,,,,,,,,,,,,
Normal (α = 1),207.37,13.73,2.34,0.0,0.16,191.46,6860.0,0.0,0.0,0.0,0.00,0.00
Partial restriction (α = 0.5),217.03,13.73,2.34,0.0,0.16,191.37,6860.0,0.0,0.0,0.0,369.08,9.76
Complete outage (α = 0),269.30,13.73,2.34,0.0,0.16,190.69,6860.0,0.0,0.0,0.0,2173.56,62.71


In [4]:
# Unmet demand on the outage day, by representative year (MWh)
(pd.DataFrame(unmet) / 1e3).round(1)

,Normal (α = 1),Partial restriction (α = 0.5),Complete outage (α = 0)
2028,0.0,19.0,384.2
2031,0.0,10.8,354.4
2037,0.0,67.0,432.2
2043,0.0,103.2,468.5
2049,0.0,169.1,534.3
